In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
import torch

from src.utils.interpolate import QuinticKernel
from src.utils.nbrs_utils import displ_fn, nearest
from torch_scatter import scatter_add

In [ ]:
Nx = 64
dim = 2
L = 2 * np.pi
is_physical = True
u_ref = 7
dt = 0.0005
write_every = 10
dt_effective = dt * write_every

N_tot = Nx**dim
dx_phys = L / Nx
l_ref = 1.0 if is_physical else dx_phys
dx = dx_phys / l_ref  # non-dimensionalize
rho_ref = 1.0  # reference density
mass = dx**dim * rho_ref
box = torch.ones(dim) * L / l_ref
pbc = True
kernel_fn = QuinticKernel(h=dx, dim=dim)
n_part_per_traj = torch.tensor([N_tot])


def get_dist_rho(r):
    edge_index = nearest(r, n_part_per_traj, pbc, box, cutoff=kernel_fn.cutoff)
    i_s, j_s = edge_index
    # print(i_s[j_s==0].sort()[0])
    # print(sum(i_s!=len(r)))
    r_i, r_j = r[i_s], r[j_s]
    dr_ij = displ_fn(r_i, r_j, box, pbc)
    dist = torch.norm(dr_ij, dim=-1)
    w_dist = kernel_fn.w(dist)
    rho = mass * scatter_add(w_dist, i_s, dim=0, dim_size=N_tot)
    return dist, rho

In [ ]:
# root100 = "../logs/train/runs/2025-06-15_23-15-04/rlt/"
# paths100 = [
#     root100 + "101_gnn",
#     root100 + "101_gnn",
#     root100 + "101_gnn_nsph1"
# ]

root = "../logs/train/runs/2025-06-15_23-15-04/rlt/"
paths = [
    root + "500_rho",
    root + "500_rho",
    root + "500_rho101",
    root + "500_rho10",
    # root + "500_rhotvf1",
]

ts = [0, 10, 499]  # time steps to plot
fig, axs = plt.subplots(4, len(ts), figsize=(20, 15), sharex="row")
is_gt = True  # apply to ground truth data
for path in paths:
    label = "dataset" if is_gt else path.split("/")[-1]
    if is_gt:
        keys = ["ground_truth_rollout", "ground_truth_u_vel"]
        style = "-"
        is_gt = False
    else:
        keys = ["predicted_rollout", "predicted_u_vel"]
        style = "--"
    kwargs = {"linestyle": style, "label": label, "histtype": "step", "density": True}

    rollout = pickle.load(open(f"{path}/rollout_0000.pkl", "rb"))
    for i_t, t in enumerate(ts):
        r_i = torch.tensor(rollout[keys[0]][t])
        r_ip1 = torch.tensor(rollout[keys[0]][t + 1])
        u_i = torch.tensor(rollout[keys[1]][t])

        # columns: frames 0; 10; 99
        # per subplot: dataset; w/ nsph; w/o nsph

        # 1st row: u magn hist
        u_mag = np.linalg.norm(u_i, axis=-1)
        axs[0, i_t].hist(u_mag, bins=50, **kwargs, range=(0, 10))

        # 2nd row: v magn hist
        v = displ_fn(r_ip1, r_i, box, pbc)
        v /= dt_effective
        v_mag = np.linalg.norm(v, axis=-1)
        axs[1, i_t].hist(v_mag, bins=50, **kwargs, range=(0, 10))

        dist, rho = get_dist_rho(r_i)
        # 3rd row: density hist
        axs[2, i_t].hist(rho, bins=50, **kwargs)  # , range=(0.9, 1.1)

        # 4th row: edge length hist
        axs[3, i_t].hist(dist / dx, bins=50, **kwargs)  # , range=(0, 3)

for ax in axs.flatten():
    ax.grid()
    ax.legend(loc="upper right")

# axs[0, 0].legend(loc='upper right')
for i, yaxis in enumerate(["u magn", "v magn", "density", "edge length"]):
    axs[i, 0].set_ylabel(yaxis)
for i, xaxis in enumerate([f"t={ts[0]}", f"t={ts[1]}", f"t={ts[2]}"]):
    axs[-2, i].set_xlabel(xaxis)

TODO:
- Cook 2007 NSPH: play with mu and mu* during inference (i.e., not during relaxation)
- rho criterion at 1.01
- 2D TGV Re=1000